# DC2 Submission Workflow

This notebook guides you through the **complete submission pipeline** for the
[DC2 Ocean Forecasting Challenge](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/):

1. **Prepare** your model predictions (customisable block)
2. **Inspect** the dataset structure
3. **Validate** conformity with the DC2 grid specification
4. **Evaluate** against observations using the full DC2 pipeline
5. **Analyse** and visualise the results
6. **Submit** to the public leaderboard

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics/blob/main/notebooks/submit.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics/main?labpath=notebooks/submit.ipynb)

## 0. Setup

If running on **Colab**, install the project first:

```bash
!pip install git+https://github.com/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics.git
```

Otherwise, make sure the `dc-env` conda environment is activated  
(see the [Quickstart guide](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/en/latest/content/quickstart.html)).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import xarray as xr

# Ensure the project root is on the Python path (works from notebooks/)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Configuration

Set the parameters for your submission here.

In [ ]:
# ── Submission metadata ──────────────────────────────────────────────
MODEL_NAME   = "MyModel"               # Short model identifier
TEAM_NAME    = "My Team"               # Team / author name
DESCRIPTION  = "Short model description"

# ── Paths ────────────────────────────────────────────────────────────
PREDICTION_DIR = Path("/tmp/dc2_submission")   # Where your predictions live
OUTPUT_DIR     = PROJECT_ROOT / "dc2_output"   # Evaluation output directory

---

## 2. Prepare your predictions  ✉️

**This is the block you should customise.** Replace the example below with
the code that loads or generates your model's predictions.

### Requirements

Your predictions must follow the DC2 grid specification:

| Dimension | Range | Step | Points |
|-----------|-------|------|--------|
| Latitude  | \u221278° to 90° | 0.25° | 672 |
| Longitude | \u2212180° to 180° | 0.25° | 1 440 |
| Depth     | 0.49 m to 5 274 m | — | 21 standard levels |
| Time      | 10 daily lead times per forecast | 1 day | 10 |

**Variables**: `zos` (SSH), `thetao` (temperature), `so` (salinity), `uo` / `vo` (velocities).

**Accepted layouts**:
- A directory of per-date `.zarr` stores (e.g. `20240101.zarr`, `20240108.zarr`, …)
- A single `.zarr` or `.nc` file covering the full period
- A directory of per-date `.nc` files

The evaluation period is **2024-01-01 to 2025-01-01** with one forecast every **7 days**.

In [ ]:
# =====================================================================
# ✏️  CUSTOMISE THIS CELL
#
# Replace the sample generator with your own data-loading code.
# The result should be a directory of Zarr/NetCDF forecast files
# written to PREDICTION_DIR.
# =====================================================================

# ─── Example 1: Generate random-noise sample (for testing) ───────────
from scripts.create_sample_submission import create_sample_dataset

create_sample_dataset(
    PREDICTION_DIR,
    variables=["zos", "thetao", "so", "uo", "vo"],
    seed=42,
)

# ─── Example 2: Load from local Zarr stores ─────────────────────────
#
# If your model already produced per-date Zarr files, just point
# PREDICTION_DIR to that directory:
#
#   PREDICTION_DIR = Path("/data/my_model/forecasts")
#
# The directory should contain files like:
#   20240101.zarr/  20240108.zarr/  20240115.zarr/  ...

# ─── Example 3: Load from a single NetCDF file ──────────────────────
#
# If you have a single file, convert it to per-date Zarr stores:
#
#   import pandas as pd
#   ds = xr.open_dataset("/data/my_model/predictions.nc")
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   for date, group in ds.groupby("time.date"):
#       store_path = PREDICTION_DIR / f"{pd.Timestamp(date):%Y%m%d}.zarr"
#       group.to_zarr(str(store_path), mode="w", consolidated=True)

# ─── Example 4: Download from S3 (Wasabi / AWS) ─────────────────────
#
#   import s3fs
#   fs = s3fs.S3FileSystem(
#       key="YOUR_KEY",
#       secret="YOUR_SECRET",
#       client_kwargs={"endpoint_url": "https://s3.eu-west-2.wasabisys.com"},
#   )
#   remote_files = fs.ls("my-bucket/DC2/MyModel/")
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   for remote in remote_files:
#       local = PREDICTION_DIR / Path(remote).name
#       fs.get(remote, str(local), recursive=True)

# ─── Example 5: Download from Copernicus Marine ─────────────────────
#
#   import copernicusmarine as cmc
#   ds = cmc.open_dataset(
#       dataset_id="cmems_mod_glo_phy-all_anfc_0.083deg_P1D-m",
#       variables=["zos", "thetao", "so", "uo", "vo"],
#       minimum_longitude=-180, maximum_longitude=180,
#       minimum_latitude=-78, maximum_latitude=90,
#       start_datetime="2024-01-01", end_datetime="2024-01-11",
#   )
#   # Regrid to 0.25° and save as Zarr stores (see data docs for details)

## 3. Inspect the dataset

Open one of the generated forecast files and verify its structure.

In [ ]:
# List prediction files
zarr_files = sorted(PREDICTION_DIR.glob("*.zarr"))
nc_files   = sorted(PREDICTION_DIR.glob("*.nc"))
pred_files = zarr_files or nc_files

print(f"Found {len(pred_files)} forecast files in {PREDICTION_DIR}:")
for f in pred_files[:10]:
    print(f"  {f.name}")
if len(pred_files) > 10:
    print(f"  ... and {len(pred_files) - 10} more")

In [ ]:
# Open the first file
if pred_files[0].suffix == ".zarr" or pred_files[0].is_dir():
    ds = xr.open_zarr(str(pred_files[0]))
else:
    ds = xr.open_dataset(str(pred_files[0]))

print(f"Latitude:   {ds.lat.values[0]:.2f} to {ds.lat.values[-1]:.2f}  ({len(ds.lat)} pts, step {np.diff(ds.lat.values[:2])[0]:.2f}°)")
print(f"Longitude:  {ds.lon.values[0]:.2f} to {ds.lon.values[-1]:.2f}  ({len(ds.lon)} pts, step {np.diff(ds.lon.values[:2])[0]:.2f}°)")
print(f"Depth:      {ds.depth.values[0]:.1f} m to {ds.depth.values[-1]:.1f} m  ({len(ds.depth)} levels)")
print(f"Time:       {ds.time.values[0]} to {ds.time.values[-1]}  ({len(ds.time)} steps)")
print(f"Variables:  {list(ds.data_vars)}")
ds

## 4. Validate against the DC2 specification

Before running the full evaluation, check that your predictions conform to
the expected grid, variables, and temporal coverage.

In [ ]:
import pandas as pd

# ── DC2 grid specification ────────────────────────────────────────────
DC2_LAT   = np.arange(-78.0, 90.0, 0.25)
DC2_LON   = np.arange(-180.0, 180.0, 0.25)
DC2_DEPTH = np.array([
    0.494025, 47.37369, 92.32607, 155.8507, 222.4752,
    318.1274, 380.213, 453.9377, 541.0889, 643.5668,
    763.3333, 902.3393, 1245.291, 1684.284, 2225.078,
    3220.820, 3597.032, 3992.484, 4405.224, 4833.291, 5274.784,
])
DC2_VARIABLES = ["zos", "thetao", "so", "uo", "vo"]
DC2_START = pd.Timestamp("2024-01-01")
DC2_END   = pd.Timestamp("2025-01-01")
N_DAYS_FORECAST = 10
N_DAYS_INTERVAL = 7
MAX_NAN_FRACTION = 0.10


def validate_submission(prediction_dir: Path) -> list[str]:
    """Run conformity checks and return a list of issues (empty = all good)."""
    issues = []

    files = sorted(prediction_dir.glob("*.zarr")) or sorted(prediction_dir.glob("*.nc"))
    if not files:
        issues.append("❌ No .zarr or .nc files found in prediction directory.")
        return issues

    # Check temporal coverage
    expected_dates = pd.date_range(DC2_START, DC2_END, freq=f"{N_DAYS_INTERVAL}D")
    print(f"Expected {len(expected_dates)} forecast init dates, found {len(files)} files.")
    if len(files) < len(expected_dates):
        issues.append(f"⚠️ Expected {len(expected_dates)} files, found {len(files)}.")

    # Check first file in detail
    f0 = files[0]
    ds0 = xr.open_zarr(str(f0)) if (f0.suffix == ".zarr" or f0.is_dir()) else xr.open_dataset(str(f0))

    # Variable check
    found_vars = set(ds0.data_vars)
    missing = set(DC2_VARIABLES) - found_vars
    if missing:
        issues.append(f"❌ Missing variables: {sorted(missing)}")
    else:
        print(f"✅ All {len(DC2_VARIABLES)} required variables present.")

    # Latitude check
    if len(ds0.lat) != len(DC2_LAT):
        issues.append(f"❌ Latitude: expected {len(DC2_LAT)} points, got {len(ds0.lat)}.")
    elif not np.allclose(ds0.lat.values, DC2_LAT, atol=0.01):
        issues.append("❌ Latitude values do not match the DC2 grid.")
    else:
        print("✅ Latitude grid matches DC2 specification.")

    # Longitude check
    if len(ds0.lon) != len(DC2_LON):
        issues.append(f"❌ Longitude: expected {len(DC2_LON)} points, got {len(ds0.lon)}.")
    elif not np.allclose(ds0.lon.values, DC2_LON, atol=0.01):
        issues.append("❌ Longitude values do not match the DC2 grid.")
    else:
        print("✅ Longitude grid matches DC2 specification.")

    # Depth check
    if len(ds0.depth) != len(DC2_DEPTH):
        issues.append(f"❌ Depth: expected {len(DC2_DEPTH)} levels, got {len(ds0.depth)}.")
    elif not np.allclose(ds0.depth.values, DC2_DEPTH, atol=0.5):
        issues.append("❌ Depth values do not match the DC2 standard levels.")
    else:
        print("✅ Depth levels match DC2 specification.")

    # Time / lead-time check
    if len(ds0.time) != N_DAYS_FORECAST:
        issues.append(f"❌ Time dimension: expected {N_DAYS_FORECAST} lead times, got {len(ds0.time)}.")
    else:
        print(f"✅ Time dimension has {N_DAYS_FORECAST} lead times.")

    # NaN fraction check (quick — first file only)
    for var in DC2_VARIABLES:
        if var in ds0.data_vars:
            nan_frac = float(ds0[var].isnull().mean().values)
            status = "✅" if nan_frac <= MAX_NAN_FRACTION else "❌"
            print(f"  {status} {var}: NaN fraction = {nan_frac:.2%}")
            if nan_frac > MAX_NAN_FRACTION:
                issues.append(f"❌ Variable '{var}' has {nan_frac:.1%} NaN (max {MAX_NAN_FRACTION:.0%}).")

    ds0.close()
    return issues


issues = validate_submission(PREDICTION_DIR)

if issues:
    print(f"\n\u2501\u2501 {len(issues)} issue(s) found \u2501\u2501")
    for issue in issues:
        print(f"  {issue}")
else:
    print("\n\u2705 All checks passed \u2014 your submission is DC2-compliant!")

## 5. Run the full evaluation pipeline

The DC2 evaluation computes metrics (RMSD, etc.) by comparing your
predictions against five independent observation / reanalysis datasets:
**GLORYS12**, **Argo profiles**, **SARAL**, **Jason-3**, and **SWOT**.

The pipeline is orchestrated by `dc2/evaluate.py`, which:
1. Downloads observation data from the Wasabi S3 bucket
2. Aligns predictions with observations in space and time
3. Computes per-lead-time and per-depth metrics
4. Produces `results_<model>.json` and `results_<model>_per_bins.jsonl.gz`

> **Note:** The full evaluation downloads several GB of observation data and
> takes significant time (typically 1–2 h on a 16-core machine with 64 GB RAM).
> Make sure you have enough disk space (~20 GB) and a stable internet connection.

### How it works

By default, the pipeline evaluates the **GloNet** baseline model (configured in
`dc2/config/dc2_wasabi.yaml`). To evaluate **your own model**, you have two options:

- **Option A (recommended):** Upload your predictions to an S3 bucket and add
  a source entry in the YAML config pointing to your model data.
- **Option B:** Modify `dc2_wasabi.yaml` to replace the `glonet` source with
  your own predictions path.

Below we demonstrate **Option B** by creating a custom YAML config
programmatically.

In [ ]:
import copy

import yaml

# Load the default DC2 config
config_path = PROJECT_ROOT / "dc2" / "config" / "dc2_wasabi.yaml"
with open(config_path) as f:
    dc2_config = yaml.safe_load(f)

# Create a custom config that replaces the glonet source with our model
custom_config = copy.deepcopy(dc2_config)

# Find and replace the glonet source entry
model_name_lower = MODEL_NAME.lower()
for i, source in enumerate(custom_config.get("sources", [])):
    if source.get("dataset") == "glonet":
        # Replace with a local-path source pointing to the user's predictions
        custom_config["sources"][i] = {
            "dataset": model_name_lower,
            "observation_dataset": False,
            "config": "s3",
            "connection_type": "local",
            "local_root": str(PREDICTION_DIR.resolve()),
            "file_pattern": "**/*.zarr",
            "keep_variables": ["zos", "thetao", "so", "uo", "vo"],
            "eval_variables": ["zos", "thetao", "so", "uo", "vo"],
            "full_day_data": True,
        }
        break

# Update dataset_references to use the custom model name
custom_config["dataset_references"] = {
    model_name_lower: ["argo_profiles", "glorys", "jason3", "saral", "swot"]
}

# Write the custom config
custom_config_path = PROJECT_ROOT / "dc2" / "config" / f"dc2_custom_{model_name_lower}.yaml"
with open(custom_config_path, "w") as f:
    yaml.dump(custom_config, f, default_flow_style=False, sort_keys=False)

print(f"Custom config written to: {custom_config_path}")
print(f"Model name: {model_name_lower}")
print(f"Prediction path: {PREDICTION_DIR.resolve()}")

### Launch the evaluation

The cell below runs `dc2/evaluate.py` with the custom config. This will:
- Download observation data (if not already cached in `dc2_output/`)
- Compare your predictions to each reference dataset
- Write results to `dc2_output/results/`

> **Tip:** If you only want to quickly test the pipeline, you can run it on a
> shorter time range by editing the `start_time` / `end_time` in the custom
> config YAML above.

> **Alternative:** To evaluate the GloNet baseline instead, change
> `--config_name` to `dc2_wasabi` below.

In [ ]:
# Run the evaluation pipeline
config_name = f"dc2_custom_{model_name_lower}"  # or "dc2_wasabi" for GloNet

cmd = [
    sys.executable, str(PROJECT_ROOT / "dc2" / "evaluate.py"),
    "--config_name", config_name,
    "--data_directory", str(OUTPUT_DIR),
]

print(f"Running: {' '.join(cmd)}")
print("This may take a while (downloading observations + computing metrics)...\n")

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("\n\u2501\u2501 STDERR \u2501\u2501")
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
else:
    print("\n\u2705 Evaluation completed successfully!")

## 6. Load and inspect results

The evaluation produces a JSON results file in `dc2_output/results/`.

In [ ]:
# Find the results file
results_file = OUTPUT_DIR / "results" / f"results_{model_name_lower}.json"

# Fallback: check for GloNet results (if running with default config)
if not results_file.exists():
    results_file = OUTPUT_DIR / "results" / "results_glonet.json"
if not results_file.exists():
    results_file = PROJECT_ROOT / "dc2" / "leaderboard_results" / "results_glonet.json"

if results_file.exists():
    with open(results_file) as f:
        data = json.load(f)
    evaluated_model = data["dataset"]
    records = data["results"][evaluated_model]
    print(f"Results loaded for model: {evaluated_model}")
    print(f"Number of evaluation records: {len(records)}")
    print(f"Reference datasets: {sorted(set(r['ref_alias'] for r in records))}")
else:
    print("No results file found. Run the evaluation cell above first.")

In [ ]:
# Parse RMSD scores grouped by reference dataset and variable
if results_file.exists():
    scores = {}  # {ref_alias: {variable: {lead_time: value}}}

    for record in records:
        ref = record["ref_alias"]
        lt  = record.get("lead_time")
        for metric in record["result"]:
            if metric["Metric"] != "rmsd":
                continue
            var = metric["Variable"]
            scores.setdefault(ref, {}).setdefault(var, {})[lt] = metric["Value"]

    # Print summary table
    for ref_name in sorted(scores):
        print(f"\n\u2501\u2501 {ref_name.upper()} \u2501\u2501")
        for var_name in sorted(scores[ref_name]):
            values = scores[ref_name][var_name]
            mean_rmsd = np.mean(list(values.values()))
            print(f"  {var_name:40s}  mean RMSD = {mean_rmsd:.4f}")

## 7. Visualise results

In [ ]:
import matplotlib.pyplot as plt

if results_file.exists() and "glorys" in scores:
    glorys = scores["glorys"]

    # Surface RMSD vs lead time
    surface_vars = [v for v in glorys if "Surface" in v]
    if surface_vars:
        fig, ax = plt.subplots(figsize=(10, 5))
        for var_name in sorted(surface_vars):
            lts = sorted(glorys[var_name].keys())
            vals = [glorys[var_name][lt] for lt in lts]
            ax.plot(lts, vals, marker="o", label=var_name)
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel("RMSD")
        ax.set_title(f"{evaluated_model} \u2014 Surface RMSD vs GLORYS12")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No GLORYS results available for plotting.")

In [ ]:
if results_file.exists() and "glorys" in scores:
    glorys = scores["glorys"]
    depth_labels = ["Surface", "50m", "200m", "550m"]

    for phys_var in ["temperature", "salinity"]:
        means = []
        for dl in depth_labels:
            key = f"{dl} {phys_var}"
            if key in glorys:
                means.append(np.mean(list(glorys[key].values())))
            else:
                means.append(np.nan)

        if any(not np.isnan(m) for m in means):
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.barh(depth_labels[::-1], means[::-1])
            ax.set_xlabel("Mean RMSD")
            ax.set_title(f"{evaluated_model} \u2014 {phys_var.capitalize()} RMSD by depth")
            ax.grid(True, alpha=0.3, axis="x")
            plt.tight_layout()
            plt.show()

In [ ]:
# Compare RMSD across all reference datasets for each variable
if results_file.exists():
    all_refs = sorted(scores.keys())
    # Collect variables that appear across multiple references
    all_vars = sorted(set(v for ref in all_refs for v in scores[ref]))
    surface_all = [v for v in all_vars if "Surface" in v or "ssh" in v.lower()]

    if surface_all and len(all_refs) > 1:
        fig, axes = plt.subplots(1, len(surface_all), figsize=(5 * len(surface_all), 4), squeeze=False)
        for ax, var_name in zip(axes[0], surface_all):
            for ref in all_refs:
                if var_name in scores[ref]:
                    lts = sorted(scores[ref][var_name].keys())
                    vals = [scores[ref][var_name][lt] for lt in lts]
                    ax.plot(lts, vals, marker=".", label=ref)
            ax.set_xlabel("Lead time (days)")
            ax.set_ylabel("RMSD")
            ax.set_title(var_name)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        fig.suptitle(f"{evaluated_model} \u2014 RMSD by reference dataset", y=1.02)
        plt.tight_layout()
        plt.show()

## 8. Prepare your leaderboard submission

To participate in the public leaderboard, you need two files produced by the
evaluation:

| File | Content |
|------|---------|
| `results_<model>.json` | Aggregated scores per variable, depth, and lead time |
| `results_<model>_per_bins.jsonl.gz` | Per-bin spatial breakdown (1°×1° RMSD maps) |

In [ ]:
results_dir = OUTPUT_DIR / "results"

json_file     = results_dir / f"results_{model_name_lower}.json"
per_bins_file = results_dir / f"results_{model_name_lower}_per_bins.jsonl.gz"

print("Files required for leaderboard submission:\n")
for path in [json_file, per_bins_file]:
    exists = "✅" if path.exists() else "❌ (not found)"
    size = f"({path.stat().st_size / 1024:.0f} KB)" if path.exists() else ""
    print(f"  {exists}  {path.name}  {size}")

if json_file.exists() and per_bins_file.exists():
    print("\n\u2705 Both files are ready. Follow the steps below to submit.")
else:
    print("\n\u26a0\ufe0f  Run the evaluation (section 5) to generate these files.")

### How to submit

1. **Fork** the repository on GitHub:  
   <https://github.com/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics>

2. **Copy** both result files into `dc2/leaderboard_results/` in your fork.

3. **Open a Pull Request** against the `main` branch with a brief description
   of your model.

4. Once merged, your model will appear on the
   [public leaderboard](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/en/latest/content/leaderboard.html).

See the full [submission guide](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/en/latest/content/submission.html)
for details.

## 9. Clean up (optional)

Remove the custom config and temporary prediction files.

In [ ]:
# Uncomment the lines below to clean up
#
# custom_config_path.unlink(missing_ok=True)
# print(f"Removed custom config: {custom_config_path}")
#
# import shutil
# if PREDICTION_DIR.exists():
#     shutil.rmtree(PREDICTION_DIR)
#     print(f"Removed predictions: {PREDICTION_DIR}")